# Đối chứng DeBERTa cho bài toán chấm CEFR từ transcript

**Người thực hiện:** Lê Văn Chiến Thắng  
**Mục tiêu:** kiểm tra lại model đã fine-tune và đo lợi ích của việc fine-tune toàn bộ encoder.

Notebook chạy ba cấu hình trên cùng dữ liệu dev:

1. `mean_baseline`: mọi bài nhận điểm trung bình của tập train.
2. `frozen_deberta`: đóng băng encoder DeBERTa, chỉ huấn luyện pooler và đầu hồi quy.
3. `full_finetune`: nạp lại weights P1/P3/P4/P5 đã huấn luyện để tính lại dự đoán.

> Đây là bảng so sánh trên **dev**, dùng để phân tích và chọn hướng. Nó chưa phải đánh giá cuối trên `eval`. Không tải corpus, transcript, dự đoán hoặc weights lên GitHub/Hugging Face.

## 0. Chuẩn bị trên Google Colab

1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Drive phải có corpus tại `MyDrive/sandi-corpus-2025`.
3. Drive phải có model cũ tại `MyDrive/aicefr-thang/deberta-results/P*/best-model`.
4. Chạy từ trên xuống. Không đổi tham số giữa các phần P.
5. Kết quả mới được ghi vào `MyDrive/aicefr-thang/deberta-ablation-frozen`, tách khỏi kết quả cũ.

In [ ]:
# Dùng đúng bộ phiên bản của lần fine-tune đã hoàn thành. Không cài lại torch.
%pip install -q "transformers==5.17.0" "datasets==5.0.1" "accelerate==1.15.0"
%pip install -q "sentencepiece==0.2.2" "scipy>=1.11" "pandas>=2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.3 MB/s eta 0:00:00


In [ ]:
import gc
import json
import math
import os
import platform
from collections import Counter, defaultdict
from pathlib import Path

import accelerate
import datasets
import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import pearsonr
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python       :", platform.python_version())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)
print("CUDA         :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU T4 trong Runtime → Change runtime type.")
print("GPU          :", torch.cuda.get_device_name(0))
print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

Python       : 3.13.15
PyTorch      : 2.11.0+cu128
Transformers : 5.17.0
Datasets     : 5.0.1
Accelerate   : 1.15.0
CUDA         : True
GPU          : Tesla T4
VRAM (GB)    : 14.6


## 1. Mount Drive và kiểm tra đầu vào

`FULL_ROOT` là kết quả fine-tune cũ. `FROZEN_ROOT` chỉ chứa kết quả đối chứng mới.

In [ ]:
from google.colab import drive

# Ép nối lại Drive để tránh lỗi FUSE: Transport endpoint is not connected.
drive.mount("/content/drive", force_remount=True)

DATA_ROOT = Path("/content/drive/MyDrive/sandi-corpus-2025")
FULL_ROOT = Path("/content/drive/MyDrive/aicefr-thang/deberta-results")
FROZEN_ROOT = Path("/content/drive/MyDrive/aicefr-thang/deberta-ablation-frozen")
FROZEN_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "microsoft/deberta-v3-base"
PARTS = [1, 3, 4, 5]
SEED = 42
MAX_LENGTH = 512
FROZEN_EPOCHS = 10
FROZEN_LEARNING_RATE = 1e-3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
SKIP_COMPLETED = True

required = [
    DATA_ROOT / "reference-materials/pre-norm",
    DATA_ROOT / "reference-materials/sla-marks",
] + [FULL_ROOT / f"P{part}/best-model/config.json" for part in PARTS]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Thiếu đầu vào:\n- " + "\n- ".join(missing))

print("Corpus          :", DATA_ROOT)
print("Full fine-tune  :", FULL_ROOT)
print("Frozen output   :", FROZEN_ROOT)

Mounted at /content/drive
Corpus          : /content/drive/MyDrive/sandi-corpus-2025
Full fine-tune  : /content/drive/MyDrive/aicefr-thang/deberta-results
Frozen output   : /content/drive/MyDrive/aicefr-thang/deberta-ablation-frozen


## 2. Đọc corpus và kiểm toán ID

P2 không xuất hiện vì corpus SLA chỉ có P1, P3, P4 và P5. Cell này không in transcript.

In [ ]:
def ctm_path(split: str, part: int) -> Path:
    return DATA_ROOT / f"reference-materials/pre-norm/{split}-sla-P{part}.ctm"


def marks_path(split: str, part: int | str) -> Path:
    part_name = f"P{part}" if isinstance(part, int) else part
    return DATA_ROOT / f"reference-materials/sla-marks/{split}-sla-{part_name}.tsv"


def read_ctm(split: str, part: int) -> tuple[dict[str, str], dict]:
    path = ctm_path(split, part)
    if not path.is_file():
        raise FileNotFoundError(path)
    words_by_id = defaultdict(list)
    malformed = 0
    part_markers = Counter()
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            columns = line.split()
            if len(columns) not in (5, 6):
                malformed += 1
                continue
            segment_id = columns[0]
            submission_id = segment_id[:12]
            words_by_id[submission_id].append(columns[4])
            if len(segment_id) > 14:
                part_markers[segment_id[14]] += 1
    return (
        {key: " ".join(words) for key, words in words_by_id.items()},
        {"malformed_lines": malformed, "part_markers": dict(part_markers)},
    )


def read_marks(split: str, part: int | str) -> dict[str, float]:
    path = marks_path(split, part)
    labels = {}
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            columns = line.rstrip("\n").split("\t")
            if len(columns) < 2:
                raise ValueError(f"Dòng nhãn lỗi tại {path}:{line_number}")
            submission_id, score_text = columns[0], columns[1]
            if submission_id in labels:
                raise ValueError(f"ID nhãn bị trùng: {submission_id}")
            score = float(score_text)
            if not 0.0 <= score <= 6.0:
                raise ValueError(f"Điểm ngoài thang 0–6 tại {path}:{line_number}")
            labels[submission_id] = score
    return labels


def load_part(split: str, part: int) -> pd.DataFrame:
    transcripts, metadata = read_ctm(split, part)
    labels = read_marks(split, part)
    if set(transcripts) != set(labels):
        raise ValueError(
            f"{split} P{part}: ID transcript/label không khớp; "
            f"difference={len(set(transcripts) ^ set(labels))}"
        )
    ids = sorted(transcripts)
    frame = pd.DataFrame({
        "submission_id": ids,
        "text": [transcripts[key] for key in ids],
        "label": [labels[key] for key in ids],
    })
    frame.attrs["ctm_meta"] = metadata
    return frame

In [ ]:
expected_counts = {
    ("train", 1): 3068, ("train", 3): 3060,
    ("train", 4): 3005, ("train", 5): 3085,
    ("dev", 1): 438, ("dev", 3): 438,
    ("dev", 4): 438, ("dev", 5): 438,
}
data_cache = {}
audit_rows = []
for part in PARTS:
    train_df = load_part("train", part)
    dev_df = load_part("dev", part)
    overlap = set(train_df.submission_id) & set(dev_df.submission_id)
    if overlap:
        raise ValueError(f"P{part}: có {len(overlap)} ID trùng train/dev")
    for split_name, frame in (("train", train_df), ("dev", dev_df)):
        meta = frame.attrs["ctm_meta"]
        if meta["malformed_lines"]:
            raise ValueError(f"{split_name} P{part}: CTM có dòng lỗi")
        if set(meta["part_markers"]) != {str(part)}:
            raise ValueError(f"{split_name} P{part}: marker sai {meta['part_markers']}")
        assert len(frame) == expected_counts[(split_name, part)]
        audit_rows.append({
            "split": split_name, "part": f"P{part}", "samples": len(frame),
            "score_mean": frame.label.mean(),
            "words_median": frame.text.str.split().str.len().median(),
            "train_dev_overlap": len(overlap),
        })
    data_cache[part] = {"train": train_df, "dev": dev_df}

dev_id_sets = [set(data_cache[p]["dev"].submission_id) for p in PARTS]
if not all(ids == dev_id_sets[0] for ids in dev_id_sets):
    raise ValueError("ID dev của P1/P3/P4/P5 không giống nhau.")
if dev_id_sets[0] != set(read_marks("dev", "overall")):
    raise ValueError("ID dev của các phần không khớp nhãn overall.")

audit_df = pd.DataFrame(audit_rows)
display(audit_df.round(3))
print("✓ Corpus và ID khớp: 12.218 train, 1.752 dev theo bốn phần.")

,split,part,samples,score_mean,words_median,train_dev_overlap
0,train,P1,3068,3.953,147.0,0
1,dev,P1,438,3.844,148.0,0
2,train,P3,3060,4.136,94.5,0
3,dev,P3,438,4.031,92.0,0
4,train,P4,3005,3.861,87.0,0
5,dev,P4,438,3.863,87.0,0
6,train,P5,3085,3.939,165.0,0
7,dev,P5,438,3.888,160.0,0


✓ Corpus và ID khớp: 12.218 train, 1.752 dev theo bốn phần.


## 3. Dataset và metric chung

Bảng cuối báo cả PCC, RMSE, MAE và bias. Bias dương nghĩa là model có xu hướng chấm cao hơn giám khảo. Metric chính để so sánh dùng dự đoán đã giới hạn về thang 0–6, giống notebook cũ; metric raw cũng được lưu để minh bạch.

In [ ]:
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


class CefrRegressionDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.ids = frame.submission_id.tolist()
        self.labels = frame.label.astype(float).tolist()
        self.encodings = tokenizer(
            frame.text.tolist(), truncation=True, max_length=MAX_LENGTH, padding=False
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = float(self.labels[index])
        return item


def regression_metrics(predictions, references) -> dict[str, float | None]:
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    references = np.asarray(references, dtype=float).reshape(-1)
    errors = predictions - references
    pcc = None if np.allclose(predictions, predictions[0]) else float(
        pearsonr(references, predictions).statistic
    )
    return {
        "pcc": pcc,
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "mae": float(np.mean(np.abs(errors))),
        "bias": float(np.mean(errors)),
    }


def compute_metrics(eval_prediction):
    return regression_metrics(eval_prediction.predictions.squeeze(), eval_prediction.label_ids)


def prediction_frame(frame, raw_predictions):
    raw = np.asarray(raw_predictions, dtype=float).reshape(-1)
    return pd.DataFrame({
        "submission_id": frame.submission_id,
        "reference": frame.label,
        "prediction_raw": raw,
        "prediction_clipped": np.clip(raw, 0.0, 6.0),
    })

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

## 4. Tính lại dự đoán từ weights đã fine-tune

Bước này chứng minh weights trong `best-model` thực sự tạo được metric đã báo cáo. Nó không huấn luyện thêm.

In [ ]:
def verify_full_model(part: int):
    model_dir = FULL_ROOT / f"P{part}/best-model"
    dev_df = data_cache[part]["dev"]
    dev_dataset = CefrRegressionDataset(dev_df)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir, dtype=torch.float32
    )
    model.float()
    args = TrainingArguments(
        output_dir=str(FROZEN_ROOT / f"verify-P{part}"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        fp16=True,
        bf16=False,
        report_to=[],
    )
    trainer = Trainer(
        model=model, args=args, data_collator=data_collator, processing_class=tokenizer
    )
    output = trainer.predict(dev_dataset)
    predictions = prediction_frame(dev_df, output.predictions.squeeze())
    metrics = regression_metrics(predictions.prediction_clipped, predictions.reference)
    raw_metrics = regression_metrics(predictions.prediction_raw, predictions.reference)

    saved_path = FULL_ROOT / f"P{part}/dev_predictions.tsv"
    if saved_path.is_file():
        saved = pd.read_csv(saved_path, sep="\t")
        joined = predictions.merge(
            saved[["submission_id", "prediction_clipped"]],
            on="submission_id", suffixes=("_rerun", "_saved"), validate="one_to_one"
        )
        max_difference = float(np.max(np.abs(
            joined.prediction_clipped_rerun - joined.prediction_clipped_saved
        )))
    else:
        max_difference = None

    result = {
        "experiment": "full_finetune", "part": f"P{part}",
        **metrics,
        **{f"raw_{key}": value for key, value in raw_metrics.items()},
        "max_difference_from_saved": max_difference,
    }
    predictions.to_csv(FROZEN_ROOT / f"full_P{part}_dev_predictions.tsv", sep="\t", index=False)
    (FROZEN_ROOT / f"full_P{part}_metrics.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    del trainer, model, dev_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return result, predictions


full_metrics = {}
full_predictions = {}
for part in PARTS:
    metrics, predictions = verify_full_model(part)
    full_metrics[part] = metrics
    full_predictions[part] = predictions
    print(
        f"P{part}: PCC={metrics['pcc']:.4f}, RMSE={metrics['rmse']:.4f}, "
        f"max_diff_saved={metrics['max_difference_from_saved']}"
    )

verification_failures = {
    f"P{part}": metrics["max_difference_from_saved"]
    for part, metrics in full_metrics.items()
    if metrics["max_difference_from_saved"] is None
    or metrics["max_difference_from_saved"] > 0.01
}
if verification_failures:
    raise RuntimeError(
        "Weights chưa tái tạo đúng file dự đoán cũ; dừng trước khi chạy đối chứng: "
        + str(verification_failures)
    )
print("✓ Bốn bộ weights tái tạo dự đoán cũ với sai số tối đa không quá 0.01.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

P1: PCC=0.7847, RMSE=0.6167, max_diff_saved=0.00390625


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

P3: PCC=0.7239, RMSE=0.5420, max_diff_saved=0.00390625


Loading weights:   0%|          | 0/202 [00:03<?, ?it/s]

P4: PCC=0.6906, RMSE=0.6756, max_diff_saved=0.00390625


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

P5: PCC=0.7999, RMSE=0.5964, max_diff_saved=0.00390625
✓ Bốn bộ weights tái tạo dự đoán cũ với sai số tối đa không quá 0.01.


## 5. Baseline điểm trung bình

Baseline đơn giản kiểm tra model có học được gì hơn việc đoán cùng một điểm cho mọi người hay không.

In [ ]:
mean_metrics = {}
mean_predictions = {}
for part in PARTS:
    train_df = data_cache[part]["train"]
    dev_df = data_cache[part]["dev"]
    prediction = np.full(len(dev_df), train_df.label.mean())
    frame = prediction_frame(dev_df, prediction)
    # PCC không xác định cho dự đoán hằng; ghi null thay vì tạo một con số giả.
    errors = frame.prediction_clipped.to_numpy() - frame.reference.to_numpy()
    metrics = {
        "experiment": "mean_baseline", "part": f"P{part}",
        "pcc": None,
        "rmse": float(np.sqrt(np.mean(errors**2))),
        "mae": float(np.mean(np.abs(errors))),
        "bias": float(np.mean(errors)),
    }
    mean_metrics[part] = metrics
    mean_predictions[part] = frame
display(pd.DataFrame(mean_metrics.values()).round(4))

,experiment,part,pcc,rmse,mae,bias
0,mean_baseline,P1,None,0.7870,0.6266,0.1095
1,mean_baseline,P3,None,0.7381,0.5983,0.1048
2,mean_baseline,P4,None,0.7267,0.5891,-0.0023
3,mean_baseline,P5,None,0.7496,0.6120,0.0506


## 6. Huấn luyện DeBERTa đóng băng encoder

`model.deberta` được đóng băng. Pooler và classifier vẫn học vì chúng chuyển biểu diễn văn bản thành một điểm. Learning rate `1e-3` phù hợp với đầu hồi quy nhỏ; giữ cùng learning rate `2e-5` của full fine-tune có thể làm baseline này chưa kịp học.

Notebook chọn checkpoint theo PCC trên dev để khớp cách làm của lần full fine-tune cũ. Vì vậy bảng này là phân tích dev, chưa phải kết quả cuối.

In [ ]:
def frozen_config(part: int):
    return {
        "experiment": "frozen_deberta",
        "part": f"P{part}",
        "model_name": MODEL_NAME,
        "seed": SEED + part,
        "epochs": FROZEN_EPOCHS,
        "learning_rate": FROZEN_LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "max_length": MAX_LENGTH,
    }


def run_frozen_part(part: int):
    part_dir = FROZEN_ROOT / f"P{part}"
    checkpoint_dir = part_dir / "checkpoints"
    metrics_path = part_dir / "metrics.json"
    predictions_path = part_dir / "dev_predictions.tsv"
    part_dir.mkdir(parents=True, exist_ok=True)
    config = frozen_config(part)

    if SKIP_COMPLETED and metrics_path.is_file() and predictions_path.is_file():
        old_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        mismatches = {key: (old_metrics.get(key), value) for key, value in config.items()
                      if old_metrics.get(key) != value}
        if mismatches:
            raise RuntimeError(f"P{part}: kết quả cũ khác cấu hình hiện tại: {mismatches}")
        print(f"P{part}: đọc kết quả frozen đã hoàn thành từ Drive.")
        return old_metrics, pd.read_csv(predictions_path, sep="\t")

    train_df = data_cache[part]["train"]
    dev_df = data_cache[part]["dev"]
    train_dataset = CefrRegressionDataset(train_df)
    dev_dataset = CefrRegressionDataset(dev_df)
    set_seed(SEED + part)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=1, problem_type="regression", dtype=torch.float32
    )
    model.float()
    for parameter in model.deberta.parameters():
        parameter.requires_grad = False
    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    forbidden = [name for name in trainable_names if name.startswith("deberta.")]
    if forbidden:
        raise RuntimeError(f"Encoder chưa đóng băng hoàn toàn: {forbidden[:5]}")
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in model.parameters())
    print(f"P{part}: trainable={trainable:,}/{total:,} ({100*trainable/total:.3f}%)")

    updates_per_epoch = math.ceil(len(train_dataset) / TRAIN_BATCH_SIZE)
    warmup_steps = max(1, math.ceil(updates_per_epoch * FROZEN_EPOCHS * 0.10))
    args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=FROZEN_EPOCHS,
        learning_rate=FROZEN_LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="pcc",
        greater_is_better=True,
        logging_steps=50,
        fp16=True,
        bf16=False,
        report_to=[],
        seed=SEED + part,
        data_seed=SEED + part,
        run_name=f"deberta-frozen-P{part}",
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_dataset, eval_dataset=dev_dataset,
        data_collator=data_collator, compute_metrics=compute_metrics, processing_class=tokenizer
    )
    last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.is_dir() else None
    trainer.train(resume_from_checkpoint=last_checkpoint)
    output = trainer.predict(dev_dataset)
    predictions = prediction_frame(dev_df, output.predictions.squeeze())
    clipped_metrics = regression_metrics(predictions.prediction_clipped, predictions.reference)
    raw_metrics = regression_metrics(predictions.prediction_raw, predictions.reference)
    metrics = {
        **config, **clipped_metrics,
        **{f"raw_{key}": value for key, value in raw_metrics.items()},
        "train_samples": len(train_df), "dev_samples": len(dev_df),
        "trainable_parameters": trainable, "total_parameters": total,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "transformers_version": transformers.__version__, "torch_version": torch.__version__,
    }
    predictions.to_csv(predictions_path, sep="\t", index=False)
    metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    trainer.save_model(str(part_dir / "best-model"))
    tokenizer.save_pretrained(str(part_dir / "best-model"))
    del trainer, model, train_dataset, dev_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return metrics, predictions


frozen_metrics = {}
frozen_predictions = {}
for part in PARTS:
    metrics, predictions = run_frozen_part(part)
    frozen_metrics[part] = metrics
    frozen_predictions[part] = predictions
    print(f"P{part}: PCC={metrics['pcc']:.4f}, RMSE={metrics['rmse']:.4f}, bias={metrics['bias']:.4f}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

P1: trainable=591,361/184,422,913 (0.321%)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

P5: PCC=0.7695, RMSE=0.4856, bias=-0.0296


## 7. Ghép overall và xuất bảng so sánh

Overall là trung bình P1/P3/P4/P5 theo cùng submission ID, đúng kế hoạch của corpus.

In [15]:
def aggregate_overall(experiment: str, predictions_by_part: dict):
    merged = None
    for part in PARTS:
        current = predictions_by_part[part][["submission_id", "prediction_clipped"]].rename(
            columns={"prediction_clipped": f"prediction_P{part}"}
        )
        merged = current if merged is None else merged.merge(
            current, on="submission_id", how="inner", validate="one_to_one"
        )
    labels = read_marks("dev", "overall")
    if set(merged.submission_id) != set(labels):
        raise ValueError(f"{experiment}: ID overall không khớp")
    merged["reference_overall"] = merged.submission_id.map(labels)
    merged["prediction_overall"] = merged[[f"prediction_P{p}" for p in PARTS]].mean(axis=1)
    metrics = {"experiment": experiment, "part": "overall", **regression_metrics(
        merged.prediction_overall, merged.reference_overall
    )}
    merged.to_csv(FROZEN_ROOT / f"{experiment}_overall_predictions.tsv", sep="\t", index=False)
    return metrics, merged


full_overall, _ = aggregate_overall("full_finetune", full_predictions)
frozen_overall, _ = aggregate_overall("frozen_deberta", frozen_predictions)
mean_overall, _ = aggregate_overall("mean_baseline", mean_predictions)

comparison_rows = []
for experiment, metrics_by_part, overall in [
    ("mean_baseline", mean_metrics, mean_overall),
    ("frozen_deberta", frozen_metrics, frozen_overall),
    ("full_finetune", full_metrics, full_overall),
]:
    comparison_rows.extend({"experiment": experiment, **metrics_by_part[p]} for p in PARTS)
    comparison_rows.append(overall)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df[["experiment", "part", "pcc", "rmse", "mae", "bias"]]
comparison_df.to_csv(FROZEN_ROOT / "comparison_dev.csv", index=False)
(FROZEN_ROOT / "comparison_dev.json").write_text(
    comparison_df.to_json(orient="records", indent=2), encoding="utf-8"
)
display(comparison_df.round(4))
display(comparison_df[comparison_df.part == "overall"].round(4))
print("✓ Đã lưu bảng so sánh tại:", FROZEN_ROOT / "comparison_dev.csv")

,experiment,part,pcc,rmse,mae,bias
0,mean_baseline,P1,NaN,0.7870,0.6266,0.1095
1,mean_baseline,P3,NaN,0.7381,0.5983,0.1048
2,mean_baseline,P4,NaN,0.7267,0.5891,-0.0023
3,mean_baseline,P5,NaN,0.7496,0.6120,0.0506
4,mean_baseline,overall,NaN,0.6706,0.5477,0.0656
5,frozen_deberta,P1,0.7496,0.5224,0.4205,0.0518
6,frozen_deberta,P3,0.6982,0.5414,0.4276,0.1181
7,frozen_deberta,P4,0.6642,0.5510,0.4439,0.0898
8,frozen_deberta,P5,0.7695,0.4856,0.3825,-0.0296
9,frozen_deberta,overall,0.8167,0.4128,0.3204,0.0575


,experiment,part,pcc,rmse,mae,bias
4,mean_baseline,overall,NaN,0.6706,0.5477,0.0656
9,frozen_deberta,overall,0.8167,0.4128,0.3204,0.0575
14,full_finetune,overall,0.8273,0.5118,0.4096,0.3456


✓ Đã lưu bảng so sánh tại: /content/drive/MyDrive/aicefr-thang/deberta-ablation-frozen/comparison_dev.csv


## 8. Cách đọc kết quả và bàn giao

Sau khi chạy xong:

1. `max_difference_from_saved` của full fine-tune nên rất nhỏ. Nếu lớn hơn `0.01`, dừng và kiểm tra weights/tokenizer có đúng lần train không.
2. So `frozen_deberta` với `full_finetune`. Chênh lệch cho thấy lợi ích của việc cập nhật encoder.
3. Chọn theo cả PCC và RMSE. Bias càng gần 0 càng ít chấm cao/thấp có hệ thống.
4. Không tuyên bố model tốt nhất chỉ từ dev. Sau khi chốt cấu hình, chạy đúng một lần trên eval.
5. Với app chấm một file audio, ưu tiên kết quả P3/P4; overall bốn phần phục vụ so sánh ở mức cả bài thi.
6. Gửi lại file `comparison_dev.csv` và output cell cuối để kiểm tra.

Không commit `best-model`, checkpoint, file dự đoán hoặc corpus. Có thể commit notebook đã xóa output nhạy cảm và bảng metric tổng hợp sau khi nhóm thống nhất.